In [1]:
import pandas as pd

f1_df = pd.read_csv(
    "../data/processed/f1_features_dataset.csv"
)

print(f1_df.shape)

(26759, 28)


In [2]:
driver_kpis = (
    f1_df
    .groupby("driver_name")
    .agg(
        total_races=("raceId", "count"),
        total_points=("points", "sum"),
        wins=("win_flag", "sum"),
        podiums=("podium_flag", "sum"),
        top10_finishes=("top10_flag", "sum"),
        avg_finish=("positionOrder", "mean"),
        avg_position_gain=("position_gain", "mean")
    )
    .reset_index()
)

In [3]:
driver_kpis["win_rate"] = (
    driver_kpis["wins"]
    /
    driver_kpis["total_races"]
    * 100
)

In [4]:
driver_kpis["podium_rate"] = (
    driver_kpis["podiums"]
    /
    driver_kpis["total_races"]
    * 100
)

In [5]:
driver_kpis["points_rank"] = (
    driver_kpis["total_points"]
    .rank(
        ascending=False,
        method="dense"
    )
)

In [6]:
consistency = (
    f1_df
    .groupby("driver_name")
    ["positionOrder"]
    .std()
    .reset_index()
)

In [7]:
consistency.columns = [
    "driver_name",
    "finish_position_std"
]

In [8]:
driver_kpis = driver_kpis.merge(
    consistency,
    on="driver_name",
    how="left"
)

In [9]:
driver_kpis.to_csv(
    "../data/processed/driver_kpis.csv",
    index=False
)

In [10]:
team_kpis = (
    f1_df
    .groupby("constructor_name")
    .agg(
        total_points=("points", "sum"),
        wins=("win_flag", "sum"),
        podiums=("podium_flag", "sum"),
        avg_finish=("positionOrder", "mean"),
        avg_position_gain=("position_gain", "mean")
    )
    .reset_index()
)

In [11]:
team_kpis["points_rank"] = (
    team_kpis["total_points"]
    .rank(
        ascending=False,
        method="dense"
    )
)

In [12]:
team_kpis.to_csv(
    "../data/processed/team_kpis.csv",
    index=False
)

In [13]:
season_driver_kpis = (
    f1_df
    .groupby(
        [
            "year",
            "driver_name"
        ]
    )
    .agg(
        season_points=("points", "sum"),
        season_wins=("win_flag", "sum"),
        season_podiums=("podium_flag", "sum")
    )
    .reset_index()
)

In [14]:
season_driver_kpis.to_csv(
    "../data/processed/season_driver_kpis.csv",
    index=False
)

In [15]:
season_team_kpis = (
    f1_df
    .groupby(
        [
            "year",
            "constructor_name"
        ]
    )
    .agg(
        season_points=("points", "sum"),
        season_wins=("win_flag", "sum"),
        season_podiums=("podium_flag", "sum")
    )
    .reset_index()
)

In [16]:
season_team_kpis.to_csv(
    "../data/processed/season_team_kpis.csv",
    index=False
)

In [17]:
driver_kpis.head()

,driver_name,total_races,total_points,wins,podiums,top10_finishes,avg_finish,avg_position_gain,win_rate,podium_rate,points_rank,finish_position_std
0,Adolf Brudes,1,0.0,0,0,0,16.000000,3.000000,0.0,0.000000,145.0,NaN
1,Adolfo Cruz,1,0.0,0,0,0,16.000000,-3.000000,0.0,0.000000,145.0,NaN
2,Adrian Sutil,128,124.0,0,0,30,14.523438,0.273438,0.0,0.000000,71.0,4.748750
3,Adrián Campos,21,0.0,0,0,1,20.857143,1.294118,0.0,0.000000,145.0,5.246768
4,Aguri Suzuki,88,8.0,0,1,17,20.659091,0.723077,0.0,1.136364,133.0,9.810722


In [18]:
team_kpis.head()

,constructor_name,total_points,wins,podiums,avg_finish,avg_position_gain,points_rank
0,AFM,0.0,0,0,20.571429,0.571429,81.0
1,AGS,2.0,0,0,25.382114,5.571429,79.0
2,ATS,7.0,0,0,18.259259,2.050420,74.0
3,Adams,0.0,0,0,27.000000,-10.000000,81.0
4,Alfa Romeo,361.0,11,28,13.396896,-1.259681,19.0


In [19]:
season_driver_kpis.head()

,year,driver_name,season_points,season_wins,season_podiums
0,1950,Alberto Ascari,11.0,0,2
1,1950,Alfredo Pián,0.0,0,0
2,1950,Bayliss Levrett,0.0,0,0
3,1950,Bill Cantrell,0.0,0,0
4,1950,Bill Holland,6.0,0,1


In [20]:
season_team_kpis.head()

,year,constructor_name,season_points,season_wins,season_podiums
0,1950,Adams,0.0,0,0
1,1950,Alfa Romeo,89.0,6,12
2,1950,Alta,0.0,0,0
3,1950,Cooper,0.0,0,0
4,1950,Deidt,10.0,0,2


In [21]:
driver_kpis.sort_values(
    "total_points",
    ascending=False
).head(10)

,driver_name,total_races,total_points,wins,podiums,top10_finishes,avg_finish,avg_position_gain,win_rate,podium_rate,points_rank,finish_position_std
523,Lewis Hamilton,356,4820.5,105,202,312,5.019663,-0.695775,29.494382,56.741573,1.0,5.551009
765,Sebastian Vettel,300,3098.0,53,122,220,7.093333,-0.724832,17.666667,40.666667,2.0,6.405836
571,Max Verstappen,209,2912.5,63,112,172,5.645933,-0.605769,30.143541,53.588517,3.0,5.997515
256,Fernando Alonso,404,2329.0,32,106,271,8.492574,0.126551,7.920792,26.237624,4.0,6.043469
502,Kimi Räikkönen,352,1873.0,21,103,232,8.491477,-0.760807,5.965909,29.261364,5.0,6.170413
831,Valtteri Bottas,247,1788.0,10,67,145,8.967611,-0.905350,4.048583,27.125506,6.0,5.997879
614,Nico Rosberg,206,1594.5,23,57,144,8.252427,-1.349515,11.165049,27.669903,7.0,6.254315
768,Sergio Pérez,283,1585.0,6,39,186,9.332155,0.529197,2.120141,13.780919,8.0,5.465812
578,Michael Schumacher,308,1566.0,91,155,227,6.879870,-2.012987,29.545455,50.324675,9.0,7.072577
132,Charles Leclerc,149,1363.0,8,43,112,7.557047,-0.986486,5.369128,28.859060,10.0,5.988031
